<a href="https://colab.research.google.com/github/ottrindade1963/analise-industrialXL/blob/main/Pipeline_Africa_MO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Análise Industrial - África e Médio Oriente

## Passos 1 a 9 com Geração Automática de Metadados

Este notebook executa o fluxo completo de análise de dados para 37 países da África e Médio Oriente, utilizando dados do Banco Mundial (WDI + WGI).

**Características:**
- Clonagem automática do repositório GitHub
- Extração automática via API (WDI + WGI)
- Agregação INNER JOIN + Dados Sintéticos (500 anos)
- Engenharia de Features avançada (lags, MA, deltas, interações)
- 7 Modelos (5 Clássicos + 2 Bayesianos)
- Interpretabilidade SHAP + Análise Geográfica
- **Sincronização simultânea de TODOS os ficheiros no Google Drive**
- **Visualização inline de TODAS as imagens geradas em cada passo**

## 0. Configuração Inicial do Colab

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive montado em /content/drive")

In [ ]:
# Instalar dependências
!pip install -q wbgapi pmdarima xgboost shap geopandas pymc arviz tensorflow scipy
print("✓ Dependências instaladas")

In [ ]:
# Clonar repositório GitHub
!git clone https://github.com/ottrindade1963/analise-industrialXL.git /content/repo
print("✓ Repositório clonado do GitHub")

In [ ]:
import os
import sys
import time
import shutil
import glob
from datetime import datetime
from pathlib import Path
from IPython.display import display, Image, HTML

# Configurar paths
REPO_DIR = '/content/repo'  # Repositório clonado do GitHub
PIPELINE_DIR = os.path.join(REPO_DIR, 'pipeline_africa_mo')  # Subdiretório do pipeline
DRIVE_DIR = '/content/drive/MyDrive/pipeline_africa_mo_resultados'  # Resultados no Drive

# Verificar se o pipeline está no subdiretório ou na raiz
if not os.path.exists(PIPELINE_DIR):
    PIPELINE_DIR = REPO_DIR  # Se estiver na raiz

os.chdir(PIPELINE_DIR)
sys.path.insert(0, PIPELINE_DIR)

# Criar diretório de resultados no Drive
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"✓ Diretório de trabalho: {PIPELINE_DIR}")
print(f"✓ Diretório de resultados (Drive): {DRIVE_DIR}")
print(f"\n  Ficheiros do pipeline:")
for f in sorted(os.listdir(PIPELINE_DIR)):
    if f.endswith('.py') or f.endswith('.md'):
        print(f"    {f}")

In [ ]:
# Função auxiliar para sincronizar TODOS os ficheiros gerados com o Drive
def sincronizar_todos_drive():
    """
    Sincroniza TODOS os diretórios de resultados com o Google Drive.
    Chamada após cada passo para backup simultâneo.
    """
    diretorio_saida = [
        'dados_brutos',
        'dados_limpos',
        'dados_agregados',
        'dados_sinteticos',
        'dados_engenharia',
        'eda_brutos',
        'eda_agregados',
        'eda_engenharia',
        'modelos_treinados',
        'resultados_avaliacao',
        'analise_estrategias',
        'shap_analysis',
        'analise_geografica',
        'analise_avancada',
        'metadados'
    ]

    ficheiros_sincronizados = 0

    for dir_name in diretorio_saida:
        origem = os.path.join(PIPELINE_DIR, dir_name)
        if not os.path.exists(origem):
            continue

        destino = os.path.join(DRIVE_DIR, dir_name)
        os.makedirs(destino, exist_ok=True)

        # Sincronizar ficheiros (incluindo subdirectórios)
        for root, dirs, files in os.walk(origem):
            rel_path = os.path.relpath(root, origem)
            dest_root = os.path.join(destino, rel_path) if rel_path != '.' else destino
            os.makedirs(dest_root, exist_ok=True)
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(dest_root, f)
                shutil.copy2(src, dst)
                ficheiros_sincronizados += 1

    return ficheiros_sincronizados


def mostrar_imagens(diretorios, titulo=None):
    """
    Mostra inline TODAS as imagens PNG encontradas nos directórios especificados.
    Pesquisa recursivamente em subdirectórios.

    Args:
        diretorios: lista de caminhos absolutos ou relativos a PIPELINE_DIR
        titulo: título opcional para o bloco de imagens
    """
    imagens = []
    for d in diretorios:
        # Resolver caminho absoluto
        if not os.path.isabs(d):
            d = os.path.join(PIPELINE_DIR, d)
        if not os.path.exists(d):
            continue
        # Buscar PNGs recursivamente
        for root, dirs, files in os.walk(d):
            for f in sorted(files):
                if f.lower().endswith('.png'):
                    imagens.append(os.path.join(root, f))

    if not imagens:
        print("  (Nenhuma imagem gerada neste passo)")
        return

    if titulo:
        display(HTML(f'<h3>📊 {titulo} ({len(imagens)} imagens)</h3>'))
    else:
        display(HTML(f'<h3>📊 Visualizações geradas ({len(imagens)} imagens)</h3>'))

    for img_path in imagens:
        # Mostrar nome do ficheiro como legenda
        nome_ficheiro = os.path.basename(img_path)
        subdir = os.path.basename(os.path.dirname(img_path))
        display(HTML(f'<hr><b>{subdir}/{nome_ficheiro}</b>'))
        display(Image(filename=img_path, width=900))

    print(f"\n  ✓ {len(imagens)} imagens visualizadas")


print("✓ Funções auxiliares definidas (sincronização + visualização de imagens)")

---
## 1. Extração de Dados (WDI + WGI)

In [ ]:
print("\n" + "="*70)
print("  PASSO 1: EXTRAÇÃO DE DADOS VIA API")
print("="*70)

t0 = time.time()
from passo1_extracao import executar_passo1
executar_passo1()

tempo_p1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 1 não gera imagens (apenas CSVs de dados brutos)

---
## 2. EDA dos Dados Brutos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2: EDA DOS DADOS BRUTOS")
print("="*70)

t0 = time.time()
from passo2_eda_brutos import executar_passo2
executar_passo2()

tempo_p2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2
mostrar_imagens(['eda_brutos'], titulo='Passo 2 - EDA Dados Brutos')

---
## 2.1. Limpeza de Dados

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.1: LIMPEZA DE DADOS")
print("="*70)

t0 = time.time()
from passo2_1_limpeza import executar_passo2_1
executar_passo2_1()

tempo_p2_1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2.1 (EDA WDI + WGI limpos)
mostrar_imagens(
    [os.path.join('dados_limpos', 'eda_wdi'), os.path.join('dados_limpos', 'eda_wgi')],
    titulo='Passo 2.1 - EDA após Limpeza (WDI + WGI)'
)

---
## 2.2. Agregação INNER JOIN + Dados Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.2: AGREGAÇÃO INNER JOIN + DADOS SINTÉTICOS (500 ANOS)")
print("="*70)

t0 = time.time()
from passo2_2_agregacao_sinteticos import executar_passo2_2
executar_passo2_2()

tempo_p2_2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 2.2 não gera imagens (apenas CSVs de dados agregados e sintéticos)

---
## 2.3. EDA Agregados + Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.3: EDA AGREGADOS + SINTÉTICOS")
print("="*70)

t0 = time.time()
from passo2_3_eda_agregados import executar_passo2_3
executar_passo2_3()

tempo_p2_3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2.3
mostrar_imagens(['eda_agregados'], titulo='Passo 2.3 - EDA Agregados e Sintéticos')

---
## 3. Engenharia de Features

In [ ]:
print("\n" + "="*70)
print("  PASSO 3: ENGENHARIA DE FEATURES AVANÇADA")
print("="*70)

t0 = time.time()
from passo3_engenharia_features import executar_passo3
executar_passo3()

tempo_p3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")


In [ ]:
# Visualizar imagens geradas no Passo 3 (PCA e análise de features)
mostrar_imagens(['eda_engenharia'], titulo='Passo 3 - Engenharia de Features (PCA e Análise)')

---
## 4. Treinamento de 7 Modelos

In [ ]:
print("\n" + "="*70)
print("  PASSO 4: TREINAMENTO DE 7 MODELOS")
print("  (5 Clássicos: RF, XGBoost, GradientBoosting, SARIMAX, LSTM)")
print("  (2 Bayesianos: PartialPooling, CompletePooling)")
print("="*70)

t0 = time.time()
from passo4_treino_modelos import executar_passo4
executar_passo4()

tempo_p4 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p4:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 4 não gera imagens (apenas modelos .pkl e metadados)

---
## 5. Avaliação de Performance

In [ ]:
print("\n" + "="*70)
print("  PASSO 5: AVALIAÇÃO DE PERFORMANCE")
print("="*70)

t0 = time.time()
from passo5_avaliacao import executar_passo5
executar_passo5()

tempo_p5 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p5:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 5
mostrar_imagens(['resultados_avaliacao'], titulo='Passo 5 - Avaliação de Performance')

### **SEGUNDA PARTE (OS FICHEIROS TEMPORÁRIOS PRECISAM SER RESTURADOS**

**Instalando dependência**

In [ ]:
!pip install gdown -q
!pip install ruptures


**FORÇAR NOVA CLONAGEM**

In [ ]:
import os

# 1. Vai para um diretório seguro (o /content sempre existe)
os.chdir('/content')

# 2. Remove qualquer resquício antigo (ignora erros se não existir)
!rm -rf /content/repo

# 3. Clona novamente
!git clone https://github.com/ottrindade1963/analise-industrialXL.git /content/repo

# 4. Entra no diretório clonado
os.chdir('/content/repo')

# 5. Verifica o commit desejado (use o hash real, sem <>)
!git checkout 434a91d

print("✓ Repositório clonado e posicionado no commit 434a91d")

**RESTAURAR FICHEIROS DO DRIVE**

In [ ]:
import os
import urllib.request

os.makedirs('/content/repo/pipeline_africa_mo/dados_engenharia', exist_ok=True)
os.makedirs('/content/repo/pipeline_africa_mo/modelos', exist_ok=True)

files_to_download = {
    '/content/repo/pipeline_africa_mo/dados_engenharia/agregado_features.csv': '1CaMqimq95jdytm6E2jQL9d-MxTIc9m0D',
    '/content/repo/pipeline_africa_mo/dados_engenharia/wdi_limpo_features.csv': '1u4F8Yie3dALIGtqpcV2eEX0bbuy5xdwy',
    '/content/repo/pipeline_africa_mo/dados_engenharia/wdi_sintetico_features.csv': '1nZmubk6frjFXHWt2YoDMH4Zb6lUvlVog',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_RandomForest.pkl': '1-PMLqrCACC4rvquBXUc8Oa4f0JQ0kNXD',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_XGBoost.pkl': '1HmA9j2vjOY5Ch7LMTcGeRjNEOBEwLijF',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_GradientBoosting.pkl': '1o3falNn2CERgeriJmizdBoyMXeFJRJi2',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_LSTM.pkl': '1WtjjrMCQXdqxANiuzm-d8uzAWZN4sFsQ',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_SARIMAX.pkl': '1hg-rx5hjyMTtEIV2mOEq28AU4hWBvLeI',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_Bayes_PartialPooling.pkl': '1m7_LPBZ-heYkI5qPs5BwFVgPMEsM57f7',
    '/content/repo/pipeline_africa_mo/modelos/modelo_Agregado_Bayes_CompletePooling.pkl': '1NGd4GGBQuUblkA44hy1UqFQJoJ9mGCXp',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_RandomForest.pkl': '1RiRLsFZ9Ng8pLQK6Itn9pi2xCU94StDe',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_XGBoost.pkl': '1JumCqePIh3I2ytKjblfE0oQ29UqoRr5y',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_GradientBoosting.pkl': '1G564nDLbxLU7Ze19fnDr2JsvDNM6NqHP',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_LSTM.pkl': '1fWS0hv3DVB_Qt20AXS_DPrb7C898WY4A',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_SARIMAX.pkl': '1DzsLJ_qHXzQvtQYNoE1a48gogdZKXlQT',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_Bayes_PartialPooling.pkl': '1j5j0SEXuw7MglAsrs7lW3M4qOmKr_Xcx',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Limpo_Bayes_CompletePooling.pkl': '1OhPSvEmg1RzZyMIfumIlsdayQ8wvPSAq',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_RandomForest.pkl': '1PWkNzNLTmFFinoGMurbTg82t0ApIcxqY',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_XGBoost.pkl': '15lX3PwHMSFT5X5oLOG5KpHzuMi4tx1Ee',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_GradientBoosting.pkl': '1w70rZvXjmdqVjd44TDsnEkQszYMUIHgM',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_LSTM.pkl': '1CWvjC4uNqvAEsX6udWmoED_UlJM0qdIJ',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_SARIMAX.pkl': '1Ran84IJ_vpDKoAexneNATLEaFcrUbgVn',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_Bayes_PartialPooling.pkl': '1wtd28C2ti-tr9rn_Wzfq4hbiXyP6GQ7l',
    '/content/repo/pipeline_africa_mo/modelos/modelo_WDI_Sintetico_Bayes_CompletePooling.pkl': '1li5I-Tcj8uZeBrt14rRBaoSsWz6REClE',
}

print("Baixando ficheiros...")
for destination, file_id in files_to_download.items():
    url = f"https://drive.google.com/uc?id={file_id}&export=download"
    try:
        urllib.request.urlretrieve(url, destination )
        print(f"✓ {os.path.basename(destination)}")
    except Exception as e:
        print(f"✗ {os.path.basename(destination)}: {e}")

print("\n✓ Concluído!")



**VERIFICAÇÃO E ORDENAÇÃO DOS CAMINHO PARA O PASSO6**

In [ ]:
import os
import sys

# Corrigir caminhos para o Colab
sys.path.insert(0, '/content/repo/pipeline_africa_mo')
import config_global as config

# Sobrescrever caminhos
config.DADOS_ENGENHARIA_DIR = '/content/repo/pipeline_africa_mo/dados_engenharia'
config.MODELOS_DIR = '/content/repo/pipeline_africa_mo/modelos'
config.ESTRATEGIAS_DIR = '/content/repo/pipeline_africa_mo/analise_estrategias'

# Verificar
print("Caminhos corrigidos:")
print(f"  Dados: {config.DADOS_ENGENHARIA_DIR}")
print(f"  Modelos: {config.MODELOS_DIR}")

# Agora execute o passo 6
from passo6_estrategias import executar_passo6
executar_passo6()


---
## 6. Análise de Estratégias

In [ ]:
import time
import os
import sys

print("\n" + "="*70)
print("  PASSO 6: ANÁLISE DE ESTRATÉGIAS")
print("="*70)

t0 = time.time()
from passo6_estrategias import executar_passo6
executar_passo6()

tempo_p6 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p6:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
#n_sync = sincronizar_todos_drive()
#print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

CÓDIGO PARA FORÇAR A VISUALIZAÇÃO NESTA SESSÃO

In [ ]:
import os
from IPython.display import display, Image, HTML

def mostrar_imagens(diretorios, titulo=None):
    """
    Mostra inline TODAS as imagens PNG encontradas nos directórios especificados.
    Compatível com Colab e sandbox.

    Args:
        diretorios: Lista de nomes de directórios (ex: ['analise_estrategias', 'shap_analysis'])
        titulo: Título opcional para a galeria
    """
    PIPELINE_DIR = '/content/repo/pipeline_africa_mo'  # Ajuste se necessário

    imagens = []

    for d in diretorios:
        # Resolver caminho absoluto
        if not os.path.isabs(d):
            d = os.path.join(PIPELINE_DIR, d)

        if not os.path.exists(d):
            print(f"  ⚠️ Directório não encontrado: {d}")
            continue

        # Buscar PNGs recursivamente
        for root, dirs, files in os.walk(d):
            for f in sorted(files):
                if f.lower().endswith('.png'):
                    imagens.append(os.path.join(root, f))

    if not imagens:
        print("  (Nenhuma imagem gerada neste passo)")
        return

    # Mostrar título
    if titulo:
        display(HTML(f'<h2>📊 {titulo}</h2>'))
        display(HTML(f'<p><b>Total: {len(imagens)} imagens</b></p>'))
    else:
        display(HTML(f'<h2>📊 Visualizações geradas ({len(imagens)} imagens)</h2>'))

    # Mostrar cada imagem
    for i, img_path in enumerate(imagens, 1):
        nome_ficheiro = os.path.basename(img_path)
        subdir = os.path.basename(os.path.dirname(img_path))

        display(HTML(f'<hr><p><b>[{i}/{len(imagens)}] {subdir}/{nome_ficheiro}</b></p>'))
        display(Image(filename=img_path, width=900))

    print(f"\n✓ {len(imagens)} imagens visualizadas com sucesso!")

print("✓ Função mostrar_imagens() definida com sucesso!")


In [ ]:
# Visualizar Passo 6 com informações
print("\n" + "="*70)
print("  VISUALIZANDO PASSO 6 - ANÁLISE DE ESTRATÉGIAS")
print("="*70)

mostrar_imagens(['analise_estrategias'], titulo='Passo 6 - Análise de Estratégias')

print("\n✓ Visualização concluída!")


---
## 7. Interpretabilidade (SHAP)

In [ ]:
import os
import shutil
from datetime import datetime

def sincronizar_todos_drive():
    """Sincroniza todos os ficheiros de resultados com Google Drive"""
    PIPELINE_DIR = '/content/repo/pipeline_africa_mo'
    DRIVE_DIR = '/content/drive/MyDrive/pipeline_africa_mo_resultados'

    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Directórios a sincronizar
    diretorios = [
        'eda_brutos', 'eda_agregados', 'eda_engenharia',
        'resultados_avaliacao', 'analise_estrategias',
        'shap_analysis', 'analise_geografica', 'analise_avancada',
        'metadados'
    ]

    n_sync = 0
    for d in diretorios:
        src = os.path.join(PIPELINE_DIR, d)
        dst = os.path.join(DRIVE_DIR, d)

        if not os.path.exists(src):
            continue

        # Copiar directório
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

        # Contar ficheiros
        for root, dirs, files in os.walk(dst):
            n_sync += len(files)

    return n_sync

print("✓ Função sincronizar_todos_drive() definida com sucesso!")


**CÓDIGO DE SINCRONIZAÇÃO**

In [ ]:
# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")


In [ ]:
print("\n" + "="*70)
print("  PASSO 7: INTERPRETABILIDADE (SHAP)")
print("="*70)

t0 = time.time()
from passo7_shap import executar_passo7
executar_passo7()

tempo_p7 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p7:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 7
mostrar_imagens(['shap_analysis'], titulo='Passo 7 - SHAP Interpretabilidade')

---
## 8. Análise Geográfica

In [ ]:
print("\n" + "="*70)
print("  PASSO 8: ANÁLISE GEOGRÁFICA")
print("="*70)

t0 = time.time()
from passo8_geografica import executar_passo8
executar_passo8()

tempo_p8 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p8:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 8
mostrar_imagens(['analise_geografica'], titulo='Passo 8 - Análise Geográfica')

---
## 9. Análises Avançadas

In [ ]:
print("\n" + "="*70)
print("  PASSO 9: ANÁLISES AVANÇADAS")
print("="*70)

t0 = time.time()
from passo9_avancada import executar_passo9
executar_passo9()

tempo_p9 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p9:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 9
mostrar_imagens(['analise_avancada'], titulo='Passo 9 - Análises Avançadas')

**10. INOVAÇÕES**

In [ ]:
print("\n" + "="*70)
print("  PASSO 10: INOVAÇÕES DE MESTRADO")
print("="*70)

t0 = time.time()
from passo10_inovacoes_mestrado import executar_passo10
executar_passo9()

tempo_p10 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p10:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

**visualizar os gráficos**

In [ ]:
from IPython.display import Image, display
import os

# Listar todos os PNGs gerados
pasta = '/content/nome_do_projecto/inovacoes_mestrado/'
for f in sorted(os.listdir(pasta)):
    if f.endswith('.png'):
        print(f"\n{'='*50}")
        print(f"  {f}")
        print(f"{'='*50}")
        display(Image(filename=os.path.join(pasta, f), width=800))


---
## Resumo Final

In [ ]:
print("\n" + "="*70)
print("  ✓ PIPELINE COMPLETO EXECUTADO COM SUCESSO!")
print("="*70)

# Resumo de tempos
tempos = {
    'Passo 1 (Extração)': tempo_p1,
    'Passo 2 (EDA Brutos)': tempo_p2,
    'Passo 2.1 (Limpeza)': tempo_p2_1,
    'Passo 2.2 (Agregação+Sintéticos)': tempo_p2_2,
    'Passo 2.3 (EDA Agregados)': tempo_p2_3,
    'Passo 3 (Features)': tempo_p3,
    'Passo 4 (Treino)': tempo_p4,
    'Passo 5 (Avaliação)': tempo_p5,
    'Passo 6 (Estratégias)': tempo_p6,
    'Passo 7 (SHAP)': tempo_p7,
    'Passo 8 (Geográfica)': tempo_p8,
    'Passo 9 (Avançada)': tempo_p9,
}

print("\n  Tempos de Execução:")
for passo, tempo in tempos.items():
    print(f"    {passo}: {tempo:.1f}s")

tempo_total = sum(tempos.values())
print(f"\n  ⏱ TEMPO TOTAL: {tempo_total:.1f}s ({tempo_total/60:.1f}m)")

print(f"\n  📁 Resultados:")
print(f"     Local (Colab): {PIPELINE_DIR}")
print(f"     Drive: {DRIVE_DIR}")

print(f"\n  📊 Ficheiros gerados:")
total_ficheiros = 0
total_imagens = 0
for d in sorted(os.listdir(PIPELINE_DIR)):
    full = os.path.join(PIPELINE_DIR, d)
    if os.path.isdir(full) and not d.startswith('.'):
        n_files = 0
        n_imgs = 0
        for root, dirs, files in os.walk(full):
            for f in files:
                n_files += 1
                if f.lower().endswith('.png'):
                    n_imgs += 1
        if n_files > 0:
            img_info = f' ({n_imgs} imagens)' if n_imgs > 0 else ''
            print(f"     📁 {d}/  ({n_files} ficheiros{img_info})")
            total_ficheiros += n_files
            total_imagens += n_imgs

print(f"\n  ✓ Total: {total_ficheiros} ficheiros gerados ({total_imagens} imagens)")
print(f"  ✓ Todos os ficheiros sincronizados com o Google Drive")

---
## Galeria Completa de Imagens (Opcional)

Executar esta célula para ver TODAS as imagens geradas pelo pipeline de uma só vez.

In [ ]:
# Galeria completa - todas as imagens do pipeline
todos_dirs_imagens = [
    'eda_brutos',
    os.path.join('dados_limpos', 'eda_wdi'),
    os.path.join('dados_limpos', 'eda_wgi'),
    'eda_agregados',
    'eda_engenharia',
    'resultados_avaliacao',
    'analise_estrategias',
    'shap_analysis',
    'analise_geografica',
    'analise_avancada'
]
mostrar_imagens(todos_dirs_imagens, titulo='GALERIA COMPLETA - Todas as Visualizações do Pipeline')

---
## Atualizar Repositório GitHub (Opcional)

In [ ]:
# OPCIONAL: Fazer push dos resultados para o GitHub
# Descomente as linhas abaixo se quiser atualizar o repositório

# !cd {REPO_DIR} && git add -A && git commit -m "Resultados pipeline $(date +%Y-%m-%d_%H:%M:%S)" && git push
# print("✓ Resultados enviados para o GitHub")